In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('./data/raw')

In [8]:
benchmark = pd.read_excel(DATA_DIR/'benchmark.xlsx')
saec = pd.read_excel(DATA_DIR/'2025-SAEC-Public-Data-File.xlsx')
ef = pd.read_csv(DATA_DIR/'ef2024a.csv')
hd = pd.read_csv(DATA_DIR/'hd2024.csv')

In [10]:
datasets = {
    "Benchmark": benchmark,
    "SAEC": saec,
    "EF": ef,
    "HD": hd
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Benchmark: (31, 14)
SAEC: (3927, 205)
EF: (113833, 73)
HD: (6072, 72)


In [12]:
for name, df in datasets.items():
    print(f"\n---{name}---")
    print(df.columns.tolist()[:14])


---Benchmark---
['uni_id', 'uni_name', 'prog_name', 'prog_application', 'prog_longevity_score', 'prog_commit_score', 'prog_support_score', 'prog_website_score', 'prog_feedback_web', 'prog_orientaion', 'prog_tech_score', 'prog_risk_score', 'prog_year-type', 'prog_feature']

---SAEC---
['UnitID', 'INSTNM', 'STABBR', 'IC2025', 'RPU_Flag', 'Primarily_Associate_Flag', 'CBSA_FIPS', 'CBSA_Name', 'Pell_Recipients', 'UG_FinAid_Cohort', 'Pell_PCT', 'AIAA_22_23', 'Asian_22_23', 'Black_22_23']

---EF---
['UNITID', 'EFALEVEL', 'LINE', 'SECTION', 'LSTUDY', 'XEFTOTLT', 'EFTOTLT', 'XEFTOTLM', 'EFTOTLM', 'XEFTOTLW', 'EFTOTLW', 'XEFAIANT', 'EFAIANT', 'XEFAIANM']

---HD---
['UNITID', 'INSTNM', 'IALIAS', 'ADDR', 'CITY', 'STABBR', 'ZIP', 'FIPS', 'OBEREG', 'CHFNM', 'CHFTITLE', 'GENTELE', 'EIN', 'UEIS']


In [13]:
for name, df in datasets.items():
    print(f"\n---{name}---")
    
    for col in df.columns:
        if any(keyword in str(col).lower() for keyword in ['unitid', 'name', 'institution', 'enroll']):
            print(col)


---Benchmark---
uni_name
prog_name

---SAEC---
UnitID
CBSA_Name
State1_Name_20_21_22
State2_Name_20_21_22
State3_Name_20_21_22
State1_Name_11_12
State2_Name_11_12
State3_Name_11_12

---EF---
UNITID

---HD---
UNITID


In [14]:
for name, df in datasets.items():
    possible_ids =[
        col for col in df.columns
        if "unitid" in str(col).lower()
    ]

    print(name, possible_ids)

Benchmark []
SAEC ['UnitID']
EF ['UNITID']
HD ['UNITID']


In [15]:
hd['UNITID'].nunique(), len(ef)

(6072, 113833)

In [16]:
hd[
    hd.astype(str)
    .apply(
        lambda row: row.str.contains(
            "University of Central Missouri",
            case = False,
            na = False
        ).any(),
        axis = 1
        
    )
]

,UNITID,INSTNM,IALIAS,ADDR,CITY,STABBR,ZIP,FIPS,OBEREG,CHFNM,...,CBSA,CBSATYPE,CSA,COUNTYCD,COUNTYNM,CNGDSTCD,LONGITUD,LATITUDE,DFRCGID,DFRCUSCG
1712,176965,University of Central Missouri,Central,108 W South Street,Warrensburg,MO,64093,29,4,Dr. Roger Best,...,47660,2,312,29101,Johnson County,2904,-93.737428,38.759111,104,2


In [21]:
hd[
    hd["INSTNM"].str.contains(
        "University of Central Missouri",
        case = False,
        na = False
    )

][["UNITID", "INSTNM"]]


,UNITID,INSTNM
1712,176965,University of Central Missouri


In [22]:
ucm_unitid = hd.loc[
    hd["INSTNM"].str.contains(
        "University of Central Missouri",
        case = False,
        na = False
    ),
    "UNITID"
].iloc[0]

print(ucm_unitid)

176965


In [24]:
ef[
    ef["UNITID"] == ucm_unitid
]

ef[ef["UNITID"] == ucm_unitid].shape

(27, 73)

In [25]:
ucm_ef = ef[ef["UNITID"] == ucm_unitid]

ucm_ef.head()


,UNITID,EFALEVEL,LINE,SECTION,LSTUDY,XEFTOTLT,EFTOTLT,XEFTOTLM,EFTOTLM,XEFTOTLW,...,XEFNRALW,EFNRALW,XEFGNDRUN,EFGNDRUN,XEFGNDRAN,EFGNDRAN,XEFGNDRUA,EFGNDRUA,XEFGNDRKN,EFGNDRKN
36142,176965,1,29,3,4,R,12857,R,5912,R,...,R,1369,R,248.0,A,NaN,R,248.0,R,12609.0
36143,176965,2,99,3,1,R,7584,R,3383,R,...,R,73,R,241.0,A,NaN,R,241.0,R,7343.0
36144,176965,3,99,3,1,R,5648,R,2674,R,...,R,47,A,NaN,A,NaN,A,NaN,A,NaN
36145,176965,4,99,3,1,R,1031,R,470,R,...,R,15,A,NaN,A,NaN,A,NaN,A,NaN
36146,176965,5,99,3,1,R,4617,R,2204,R,...,R,32,A,NaN,A,NaN,A,NaN,A,NaN


In [27]:
ucm_ef.columns.tolist()

['UNITID',
 'EFALEVEL',
 'LINE',
 'SECTION',
 'LSTUDY',
 'XEFTOTLT',
 'EFTOTLT',
 'XEFTOTLM',
 'EFTOTLM',
 'XEFTOTLW',
 'EFTOTLW',
 'XEFAIANT',
 'EFAIANT',
 'XEFAIANM',
 'EFAIANM',
 'XEFAIANW',
 'EFAIANW',
 'XEFASIAT',
 'EFASIAT',
 'XEFASIAM',
 'EFASIAM',
 'XEFASIAW',
 'EFASIAW',
 'XEFBKAAT',
 'EFBKAAT',
 'XEFBKAAM',
 'EFBKAAM',
 'XEFBKAAW',
 'EFBKAAW',
 'XEFHISPT',
 'EFHISPT',
 'XEFHISPM',
 'EFHISPM',
 'XEFHISPW',
 'EFHISPW',
 'XEFNHPIT',
 'EFNHPIT',
 'XEFNHPIM',
 'EFNHPIM',
 'XEFNHPIW',
 'EFNHPIW',
 'XEFWHITT',
 'EFWHITT',
 'XEFWHITM',
 'EFWHITM',
 'XEFWHITW',
 'EFWHITW',
 'XEF2MORT',
 'EF2MORT',
 'XEF2MORM',
 'EF2MORM',
 'XEF2MORW',
 'EF2MORW',
 'XEFUNKNT',
 'EFUNKNT',
 'XEFUNKNM',
 'EFUNKNM',
 'XEFUNKNW',
 'EFUNKNW',
 'XEFNRALT',
 'EFNRALT',
 'XEFNRALM',
 'EFNRALM',
 'XEFNRALW',
 'EFNRALW',
 'XEFGNDRUN',
 'EFGNDRUN',
 'XEFGNDRAN',
 'EFGNDRAN',
 'XEFGNDRUA',
 'EFGNDRUA',
 'XEFGNDRKN',
 'EFGNDRKN']

In [28]:
ucm_ef.nunique().sort_values()

EFGNDRAN     0
UNITID       1
XEFTOTLM     1
XEFTOTLT     1
XEFAIANM     1
            ..
EFTOTLM     27
EFTOTLW     27
EFALEVEL    27
EFTOTLT     27
EFWHITM     27
Length: 73, dtype: int64

In [29]:
varying_cols = ucm_ef.columns[ucm_ef.nunique() > 1]

varying_cols.tolist()

['EFALEVEL',
 'LINE',
 'SECTION',
 'LSTUDY',
 'EFTOTLT',
 'EFTOTLM',
 'EFTOTLW',
 'EFAIANT',
 'EFAIANM',
 'EFAIANW',
 'EFASIAT',
 'EFASIAM',
 'EFASIAW',
 'EFBKAAT',
 'EFBKAAM',
 'EFBKAAW',
 'EFHISPT',
 'EFHISPM',
 'EFHISPW',
 'EFNHPIT',
 'EFNHPIM',
 'EFNHPIW',
 'EFWHITT',
 'EFWHITM',
 'EFWHITW',
 'EF2MORT',
 'EF2MORM',
 'XEF2MORW',
 'EF2MORW',
 'EFUNKNT',
 'EFUNKNM',
 'EFUNKNW',
 'EFNRALT',
 'EFNRALM',
 'EFNRALW',
 'XEFGNDRUN',
 'EFGNDRUN',
 'XEFGNDRUA',
 'EFGNDRUA',
 'XEFGNDRKN',
 'EFGNDRKN']

In [30]:
[col for col in ef.columns if any(x in col.upper() for x in ["LEVEL", "LINE", "SEX", "RACE"])]

['EFALEVEL', 'LINE']

In [32]:
ucm_ef[
    ucm_ef["EFALEVEL"].isin([1,2,12])
][
    ["UNITID", "EFALEVEL", "EFTOTLT", "EFNRALT"]
]

,UNITID,EFALEVEL,EFTOTLT,EFNRALT
36142,176965,1,12857,3412
36143,176965,2,7584,147
36148,176965,12,5273,3265


In [34]:
ef_total = ef[ef["EFALEVEL"] == 1][
    [
        "UNITID",
        "EFTOTLT",
        "EFNRALT"
    ]
].copy()

In [36]:
ef_total = ef_total.rename(
    columns={
        "EFTOTLT": "uni_total_enrollment",
        "EFNRALT": "uni_total_nonresident_enrollment"
    }
)

In [37]:
ef_total["uni_total_nonresident_pct"] = (
    ef_total["uni_total_nonresident_enrollment"] / ef_total["uni_total_enrollment"]
    * 100
)